[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/hopf/hopf.ipynb)

# The Hopf Fibration

A unit spinor of three-dimensional space points in a direction: sandwiching z with it gives a unit vector. It holds more than the direction, though: turning it on the right in the xy plane changes the spinor but not where it points. Sorting all unit spinors by the direction they point sends the three-sphere of spinors onto the two-sphere of directions, and the spinors over each direction form a circle. Those circles fill the three-sphere, and every two of them are linked once. This notebook builds the sorting map as a form with two spinor slots, finds each circle as an eigenspace of that form, draws the circles in space, measures how they link, and carries a spinor around a loop of directions, back onto its own circle but turned along it.

The state of a spin one half, such as an electron's spin or a qubit, is such a spinor, and the turn along the circle is the Berry phase of the graphene and magnetic resonance examples.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_animation
from examples.math.hopf import render

np.set_printoptions(precision=4, suppress=True)

# The geometric algebra of three-dimensional space.
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
# Spinors are the even multivectors: four real numbers, the unit ones forming the three-sphere.
Even = ga.gatype.even()
Rotor = ga.gatype.rotor()


def sphere(polar: np.ndarray, azimuth: np.ndarray) -> Vector:
    """Unit directions at the given polar angles from z and azimuths about it."""
    polar, azimuth = np.broadcast_arrays(polar, azimuth)
    return mv.vector(np.stack([np.sin(polar) * np.cos(azimuth), np.sin(polar) * np.sin(azimuth), np.cos(polar)], axis=-1))

## 1. A spinor's direction, as a form

Sandwiching z with a spinor turns z into the spinor's direction, `psi >> mv.z`. The spinor appears twice in that sandwich. Leaving it open, `Even >> mv.z` is a form with two spinor slots that returns a vector; on the same spinor twice it is that spinor's direction. Turning a spinor on the right in the xy plane, `psi * (mv.xy * angle).exp()`, does not change its direction, because z commutes with xy: the turn is a phase that the direction cannot see.

In quaternion notation a spinor reads as a unit quaternion $q$ and its direction as $q\,k\,\bar q$, the map of Hopf's 1931 paper; in matrix notation, as a two-component complex spinor $\psi$ and its Bloch vector $\psi^\dagger \boldsymbol\sigma \psi$, with the phase $e^{i\alpha}\psi$.

In [ ]:
hopf = Even >> mv.z                                                             # [] Vector <- (Even, Even)

psi = mv(Even.output_subspace, np.array([0.4, -0.3, 0.7, 0.5])).normalized()    # [] Even
direction = hopf(psi, psi)                                                      # [] Vector
# The same spinor turned on the right by a phase.
phased = psi * (mv.xy * 1.1).exp()                                              # [] Even

In [ ]:
print(hopf)
print("direction:       ", render.components(direction))
print("after the phase: ", render.components(hopf(phased, phased)))
print("psi >> z:        ", render.components(psi >> mv.z))

## 2. The fibre over a direction is an eigenspace

Pairing the form with a direction, `direction | hopf`, leaves a scalar from two spinors: how far a spinor's direction leans along the given one. Its eigenvalues are minus one twice and plus one twice, and the spinors with eigenvalue plus one are exactly those that point along the direction. So the fibre over a direction is the top eigenspace of that paired form, a plane through the origin that meets the three-sphere in a circle. One spinor of the eigenspace, turned by every phase, traces the whole circle.

In [ ]:
def fibre_start(direction: Vector) -> Even:
    """A unit spinor pointing along each direction: the top eigenvector of the paired form."""
    _, spinors = (direction | hopf).eigh()                                      # [..., 4] Even
    return spinors[..., -1]                                                     # [...] Even


def fibre(start: Even, angles: np.ndarray) -> Even:
    """The spinors pointing the same way as each start: the start turned on the right in the xy plane."""
    return start[..., None] * (mv.xy * mv.scalar(angles[:, None])).exp()        # [..., angles] Even


# The eigenvalues of the paired form for one direction.
values, _ = (direction | hopf).eigh()                                           # [4] Scalar
start = fibre_start(direction)                                                  # [] Even
circle = fibre(start, np.linspace(0.0, 2 * np.pi, 7))                           # [7] Even

In [ ]:
print("eigenvalues of direction | hopf:", values.to_array())
print("directions along the fibre:\n", render.components(hopf(circle, circle)))

## 3. The fibres in space

The three-sphere does not fit in space, but its stereographic projection from the spinor −1 does: a unit spinor goes to its bivector part over one plus its scalar part, read as the vector it is dual to. Every fibre is a great circle of the three-sphere and projects to a circle in space. The fibres over a circle of directions fill a torus, and circles of directions at different heights give nested tori. Each direction and its fibre share a colour.

In [ ]:
def stereographic(spinor: Even) -> Vector:
    """The stereographic projection of unit spinors from -1 into space."""
    return (spinor.select[2] / (1 + spinor.select[0])).dual()                   # [...] Vector


# Eighteen directions on each of three circles of latitude, and their fibres.
polars = np.pi * np.array([0.9, 0.7, 0.5])
azimuths = np.linspace(0.0, 2 * np.pi, 18, endpoint=False)
directions = sphere(polars[:, None], azimuths[None, :])                         # [circles, per_circle] Vector
fibres = fibre(fibre_start(directions), np.linspace(0.0, 2 * np.pi, 241))       # [circles, per_circle, samples] Even
projected = stereographic(fibres)                                               # [circles, per_circle, samples] Vector
render.draw_tori(directions, projected);

## 4. Every two fibres are linked once

Two closed curves in space are linked if one cannot be pulled free of the other. The linking number counts how often: the volume each pair of small segments spans with the line between them, over the cube of that line's length, summed over both curves and divided by four pi. For any two fibres it comes out one.

In vector calculus the linking number reads as Gauss's double integral, $\frac{1}{4\pi}\oint\oint \frac{(\mathbf r_1 - \mathbf r_2)\cdot(d\mathbf r_1 \times d\mathbf r_2)}{|\mathbf r_1 - \mathbf r_2|^3}$.

In [ ]:
def linking(first: Vector, second: Vector) -> Scalar:
    """Gauss's linking number of two closed polygons."""
    step_first = first[1:] - first[:-1]                                          # [n] Vector
    step_second = second[1:] - second[:-1]                                       # [m] Vector
    middle_first = 0.5 * (first[1:] + first[:-1])                                # [n] Vector
    middle_second = 0.5 * (second[1:] + second[:-1])                             # [m] Vector
    separation = middle_first[:, None] - middle_second[None, :]                 # [n, m] Vector
    # The trivector the separation spans with the two segments, as a scalar volume.
    volume = (separation ^ step_first[:, None] ^ step_second[None, :]).dual()       # [n, m] Scalar
    return (volume / (separation | separation).square_root() ** 3).sum() / (4 * np.pi)   # [] Scalar


pairs = [((0, 0), (0, 5)), ((0, 0), (1, 3)), ((0, 4), (2, 11)), ((1, 7), (2, 2))]
numbers = stack([linking(projected[a], projected[b]) for a, b in pairs])        # [pairs] Scalar

In [ ]:
print("linking numbers:", numbers.to_array())

As a direction spirals over the sphere, its fibres wind around one another and build up the nested tori; the newest fibre is drawn heavier.

In [ ]:
def sweep(frames: int, samples: int):
    """Directions spiralling from near the south pole to just above the equator, each with its projected fibre."""
    for fraction in np.linspace(0.0, 1.0, frames):
        direction = sphere(np.array(np.pi * (0.95 - 0.5 * fraction)), np.array(2 * np.pi * 4 * fraction))   # [] Vector
        yield direction, stereographic(fibre(fibre_start(direction), np.linspace(0.0, 2 * np.pi, samples + 1)))


display(Image(filename=save_animation(render.animate_sweep(sweep(90, 240)), "hopf_sweep", 60)))

## 5. A loop of directions turns a spinor along its fibre

Carry a spinor around a closed loop of directions, turning it at each step by the smallest rotation from one direction to the next. When the loop closes, the spinor points the way it started, so it is back on its own fibre, but turned along it: by a phase of half the solid angle the loop encloses. The picture shows the loop on the sphere, the carried spinor leaving its fibre, drawn grey, and coming back onto it further along.

In bra-ket notation this turn reads as the Berry phase $\gamma = \Omega/2$ of a spin one half carried around a loop of solid angle $\Omega$, and the smallest rotation from one direction to the next as parallel transport in the Berry connection, which the graphene example uses to carry its frame.

In [ ]:
def transport(directions: Vector) -> Rotor:
    """The rotors that carry a frame from the first direction to each of the others, by the smallest
    rotation from each direction to the next."""
    steps = (1 + directions[1:] * directions[:-1]).normalized()                # [steps] Rotor
    return steps.cumprod(axis=0)                                                  # [steps] Rotor


# A loop of directions at a polar angle of 2.2 radians, and the spinor carried around it.
polar = 2.2
loop = sphere(np.full(401, polar), np.linspace(0.0, 2 * np.pi, 401))            # [401] Vector
begin = fibre_start(loop[0])                                                    # [] Even
carried = transport(loop) * begin                                               # [steps] Even
# Where the spinor ends up along its fibre, as a turn from where it began.
turn = begin.reverse() * carried[-1]                                            # [] Even
half_solid_angle = np.pi * (1 - np.cos(polar))
render.draw_lift(loop, stereographic(carried), stereographic(fibre(begin, np.linspace(0.0, 2 * np.pi, 401))));

In [ ]:
print("turn along the fibre:       ", turn.kernel)
print("(mv.xy * -half_solid_angle).exp():", (mv.xy * -half_solid_angle).exp().kernel)

In [ ]:
# checks
# The form on one spinor twice is its direction, and a phase does not change it; the paired form's
# eigenvalues are -1, -1, 1, 1 and its top eigenvectors point along the direction; any two fibres link
# once; the carried spinor comes back turned by half the solid angle.
np.testing.assert_allclose((direction - (psi >> mv.z)).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((hopf(phased, phased) - direction).kernel, 0.0, atol=1e-9)
np.testing.assert_allclose(values.to_array(), [-1.0, -1.0, 1.0, 1.0], atol=1e-12)
np.testing.assert_allclose((hopf(fibres, fibres) - directions[..., None]).kernel, 0.0, atol=1e-9)
np.testing.assert_allclose(numbers.to_array(), 1.0, atol=1e-2)
np.testing.assert_allclose((turn - (mv.xy * -half_solid_angle).exp()).kernel, 0.0, atol=1e-4)